# When do BIXI stations run empty?

An analysis of BIXI Montréal's 2025 trip data (14.2 million trips) to find which stations drain or overflow, and when.

Data: [BIXI Montréal open data](https://bixi.com/en/open-data), licensed CC BY 4.0.

## 1. Loading the data

The 2025 file is a single 2.7 GB CSV. Station names and boroughs repeat millions of times, so they're loaded as categories, and coordinates as `float32`. This brings the full dataset down to 516 MB in memory.

In [1]:
import pandas as pd

path = "data/DonneesOuvertes2025_010203040506070809101112.csv"

dtypes = {
    "STARTSTATIONNAME": "category",
    "STARTSTATIONARRONDISSEMENT": "category",
    "ENDSTATIONNAME": "category",
    "ENDSTATIONARRONDISSEMENT": "category",
    "STARTSTATIONLATITUDE": "float32",
    "STARTSTATIONLONGITUDE": "float32",
    "ENDSTATIONLATITUDE": "float32",
    "ENDSTATIONLONGITUDE": "float32",
}

df = pd.read_csv(path, dtype=dtypes)
print(df.shape)
df.info(memory_usage="deep")

(14249363, 10)
<class 'pandas.DataFrame'>
RangeIndex: 14249363 entries, 0 to 14249362
Data columns (total 10 columns):
 #   Column                      Dtype   
---  ------                      -----   
 0   STARTSTATIONNAME            category
 1   STARTSTATIONARRONDISSEMENT  category
 2   STARTSTATIONLATITUDE        float32 
 3   STARTSTATIONLONGITUDE       float32 
 4   ENDSTATIONNAME              category
 5   ENDSTATIONARRONDISSEMENT    category
 6   ENDSTATIONLATITUDE          float32 
 7   ENDSTATIONLONGITUDE         float32 
 8   STARTTIMEMS                 int64   
 9   ENDTIMEMS                   float64 
dtypes: category(4), float32(4), float64(1), int64(1)
memory usage: 516.5 MB


## 2. Cleaning

Before analyzing anything, check how complete the data is.

In [2]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "percent": missing_pct})

,missing,percent
STARTSTATIONNAME,0,0.00
STARTSTATIONARRONDISSEMENT,22,0.00
STARTSTATIONLATITUDE,0,0.00
STARTSTATIONLONGITUDE,0,0.00
ENDSTATIONNAME,51823,0.36
ENDSTATIONARRONDISSEMENT,52861,0.37
ENDSTATIONLATITUDE,51823,0.36
ENDSTATIONLONGITUDE,51823,0.36
STARTTIMEMS,0,0.00
ENDTIMEMS,51822,0.36


51,823 trips (0.36%) have no recorded end station or end time. They can't count as arrivals anywhere, so they're dropped.

Some trips have a station but no borough. Checking which stations those are (below) shows most are not public stations at all.

In [3]:
no_borough = df[df["ENDSTATIONARRONDISSEMENT"].isna()]
no_borough["ENDSTATIONNAME"].value_counts().head(10)

ENDSTATIONNAME
cyclo                                           957
Cyclo PBSC                                       29
Cyclo Gervais                                    23
Hochelaga / Dézéry                               14
Test ECO5.1.1                                    10
Campus MIL (Thérèse-Lavoie Roux / Outremont)      5
10e Avenue Nord / King Est                        0
15e avenue / Dandurand                            0
15e avenue / Laurier                              0
3e avenue / Wellington                            0
Name: count, dtype: int64

In [4]:
start = df["STARTSTATIONNAME"].astype(str).str.contains("cyclo|test", case=False)
end = df["ENDSTATIONNAME"].astype(str).str.contains("cyclo|test", case=False)
internal = start | end

print(f"{internal.sum():,} internal trips")
pd.concat([df.loc[start, "STARTSTATIONNAME"], df.loc[end, "ENDSTATIONNAME"]]).astype(str).value_counts()


1,065 internal trips


cyclo                      964
Cyclo winter station        41
Cyclo PBSC                  29
Cyclo Gervais               23
Test ECO5.1.1               10
9599 (test cyclochrome)      4
Name: count, dtype: int64

In [5]:
df = df[~internal].copy()
print(f"{len(df):,} trips remain")

14,248,298 trips remain


The stations missing a borough are mostly internal: BIXI's repair workshop ("cyclo", "Cyclo PBSC", "Cyclo Gervais", "Cyclo winter station") and test stations. These are operational movements, not rider trips, so all 1,065 trips to or from them are removed. The remaining stations without a borough (e.g. Hochelaga / Dézéry, Campus MIL) are real stations and are kept.

In [6]:
before = len(df)
df = df.dropna(subset=["ENDSTATIONNAME", "ENDTIMEMS"])
print(f"Dropped {before - len(df):,} trips ({(before - len(df)) / before:.2%})")

df["start"] = pd.to_datetime(df["STARTTIMEMS"], unit="ms", utc=True).dt.tz_convert("America/Montreal")
df["end"] = pd.to_datetime(df["ENDTIMEMS"], unit="ms", utc=True).dt.tz_convert("America/Montreal")
df["duration_min"] = (df["end"] - df["start"]).dt.total_seconds() / 60

df["duration_min"].describe()

Dropped 51,823 trips (0.36%)


count    1.419648e+07
mean     1.642342e+01
std      2.907328e+02
min      8.833333e-04
25%      6.067317e+00
50%      1.051807e+01
75%      1.795072e+01
max      3.712294e+05
Name: duration_min, dtype: float64

Trip durations range from 0.05 seconds to 258 days. The median trip is 10.5 minutes. BIXI's documentation says trips under 1 minute and over 2 hours are already excluded, but that filter was not applied to the 2025 file, so it's applied here.

In [7]:
pd.set_option("display.float_format", "{:.2f}".format)

short = (df["duration_min"] < 1).sum()
long = (df["duration_min"] > 120).sum()
print(f"Under 1 min: {short:,} ({short / len(df):.2%})")
print(f"Over 2 hours: {long:,} ({long / len(df):.2%})")

Under 1 min: 184,991 (1.30%)
Over 2 hours: 44,653 (0.31%)


In [8]:
quick = df[df["duration_min"] < 1]
same = (quick["STARTSTATIONNAME"].astype(str) == quick["ENDSTATIONNAME"].astype(str)).mean()
print(f"{same:.1%} of sub-1-minute trips start and end at the same station")

95.2% of sub-1-minute trips start and end at the same station


184,991 trips (1.30%) last under a minute and 44,653 (0.31%) over two hours. 95.2% of the sub-minute trips start and end at the same station, consistent with riders immediately re-docking a bike (for example, a faulty one). Neither group reflects a real trip.

In [9]:
before = len(df)
df = df[(df["duration_min"] >= 1) & (df["duration_min"] <= 120)].copy()
print(f"Dropped {before - len(df):,} trips, {len(df):,} remain")

df["hour"] = df["start"].dt.hour
df["end_hour"] = df["end"].dt.hour
df["month"] = df["start"].dt.month
df["weekday"] = df["start"].dt.dayofweek < 5

df = df.drop(columns=["STARTTIMEMS", "ENDTIMEMS"])
df.to_parquet("data/trips_clean.parquet")
print("Saved")

Dropped 229,644 trips, 13,966,831 remain
Saved


### Cleaning summary

| Removed | Trips | Share |
|---|---|---|
| Workshop / test stations | 1,065 | 0.01% |
| No recorded end | 51,823 | 0.36% |
| Under 1 minute | 184,991 | 1.30% |
| Over 2 hours | 44,653 | 0.31% |
| **Kept** | **13,966,831** | **98.0%** |

The cleaned data is saved to `data/trips_clean.parquet`.

## 3. Exploration

In [10]:
df = pd.read_parquet("data/trips_clean.parquet")
print(f"{len(df):,} trips")

13,966,831 trips
